# Exploratory data analysis

Experimental output. Not a clinical decision.

Run with the repository root as the kernel working directory and the pinned dependencies installed. The notebook imports reusable analysis from `src/eda.py`; Plotly presentation helpers are in `dashboard/utils/visualization.py` as requested for dashboard reuse.

This is descriptive, request-level exploration of the prepared labelled cohort. Unlabelled requests are excluded from all target comparisons. No split exists or is created here; a later holdout drawn from this explored cohort is not fully blinded. Repeated requests from a patient are dependent. No hypothesis tests, imputation, fitted encoders, or model selection occur here.

All figure counts and written numerical observations are computed from the data and saved under `artifacts/eda/`. Interactive HTML files use a shared local Plotly bundle, key figures are also saved as PNGs, and CSV tables preserve counts, within-category fractions, numeric summaries, and pairwise correlation sample sizes.


In [1]:
from src.eda import build_eda_report, show_analytical_patterns, show_figures

figures = build_eda_report()

Saved 25 interactive figures to /Users/asherwu/Nexora_Care_Triage-/artifacts/eda
Labelled requests: 14,809; excluded unlabelled: 276


## Urgency distribution

The bar chart shows request counts and shares in the configured ordinal order. Unequal class sizes matter when interpreting the later count charts; the normalised category charts use each category's own denominator.


In [2]:
show_figures(figures, ["urgency_distribution"])

## Age, recorded sex, and pregnancy response

Age is derived at request time. Histograms include observed values; violins show the full observed range with embedded quartile boxes and per-urgency sample sizes. Sex and pregnancy figures pair stacked counts with within-category urgency fractions. The audit cross-tab retains pregnancy "Not applicable" separately from "No"; applicability is not inferred from recorded sex.


In [3]:
show_figures(figures, ["patient_age", "sex", "pregnancy_flag", "pregnancy_by_sex"])

## Chief complaint

Category totals are shown next to the labels. Compare the normalised composition as well as request counts, especially for less frequent complaints.


In [4]:
show_figures(figures, ["chief_complaint_category"])

## Severity, symptom duration, onset, and pain

Severity and pain follow their ordinal label order. Duration is shown on its full days scale and as log-transformed hours. Seconds are inferred from the raw magnitudes and minute alignment, not confirmed by source metadata. No tail trimming or imputation is applied. Compare the raw and transformed views before selecting assumptions for future tests.


In [5]:
show_figures(
    figures,
    [
        "symptom_severity",
        "symptom_duration_days",
        "symptom_duration_log1p_hours",
        "symptom_onset",
        "pain_level",
    ],
)

## Red flags and comorbidities

Item frequencies count each request once per reported item; a request may therefore appear in multiple rows. The denominator for overall item prevalence is all labelled requests, with prevalence tables saved alongside figures. The right panels show urgency fractions within the response row. Explicit sentinel responses and unknown responses have separate rows. Unknown counts remain missing in count-distribution plots and are never treated as known absence.


In [6]:
show_figures(figures, ["red_flags", "red_flag_count", "comorbidities", "comorbidity_count"])

## Vital distributions and missingness

Systolic and diastolic blood pressures remain separate. Distributions omit missing values and show observed sample sizes. The missingness heatmap includes all requests in its denominators and shows counts and fractions overall and within each urgency class. The broad technical screening thresholds used during preparation are recorded in `artifacts/data_profile.json`; they are not clinical normal ranges.


In [7]:
show_figures(
    figures,
    [
        "systolic_bp",
        "diastolic_bp",
        "body_temperature",
        "heart_rate",
        "respiratory_rate",
        "oxygen_saturation",
        "vital_missingness",
    ],
)

## Prior requests and numeric correlations

The history count includes only strictly earlier intake requests, including prior unlabelled requests, and excludes timestamp ties. Spearman heatmaps use pairwise complete observations without imputation. Pairwise sample counts are available in tooltips and companion CSV files. The full heatmap covers all numeric model features; the core heatmap provides a more readable view. Urgency is represented by its configured ordinal rank for descriptive correlation only and remains excluded from model features. Correlation between binary item indicators does not establish an independent relationship.


In [8]:
show_figures(
    figures, ["prior_patient_request_count", "correlations_core", "correlations_all_numeric"]
)

## Interpretation

The final written section reads values from the saved summary behind the plots. Proposed relationships are exploratory questions for subsequent testing. It also reports data quality surprises, cohort limitations, dependence between requests, and uncertainty about source units.


In [9]:
show_analytical_patterns()

## Analytical patterns

These observations describe the exported plots and their accompanying summary tables.
They nominate relationships for later testing; no significance or causal claims are made.
The cohort contains 14,809 labelled requests from
9,792 patients. Requests from the same patient are dependent.
No train/validation/test split exists; future holdouts from this explored cohort are
not fully blinded to these descriptive patterns.

- **Severity and pain:** The normalised bars show different urgency compositions.
  High/Critical shares by severity are Mild: 5.7%; Moderate: 30.8%; Severe: 84.1%;
  by pain they are No pain: 6.4%; Mild: 22.6%; Moderate: 45.9%; Severe: 78.6%. Test ordinal associations
  and check whether the two variables add overlapping information. These are
  associations with clinician-assigned labels, not evidence of clinical validity.
- **Chief complaint and onset:** High/Critical shares range from Skin Rash
  (16.5%) to Breathing Difficulty (73.9%). Onset shares are
  Sudden: 44.2%; Gradual: 23.8%; Recurring: 32.1%. Test differences in class composition,
  with adequate cell counts and adjustment for severity and related symptoms.
- **Duration and age:** The box/violin plots show per-urgency age medians of
  Low 33.65, Moderate 37.26, High 38.69, Critical 42.72 years and duration medians of
  Low 3.12, Moderate 2.77, High 1.93, Critical 0.77 days. The raw duration plot has a long
  right tail (maximum 120.00 days); the log-hours view
  helps compare the centre. Test distributional differences without assuming
  normality; account for possible confounding with complaint and onset.
- **Red flags and comorbidities:** Item bars overlap because responses may contain
  several items. High/Critical shares for red-flag responses are
  Breathing difficulty: 83.7%; Chest pain: 80.6%; Confusion: 81.9%; Loss of consciousness: 79.4%; Seizure: 83.5%; Severe allergic reaction: 68.9%; Severe bleeding: 80.0%; Stroke signs: 73.0%; Thoughts of self-harm: 100.0%; No red flags: 18.1%; Unknown (missing response): 29.0%.
  For comorbidity responses they are
  Asthma: 38.0%; Chronic kidney disease: 41.9%; Diabetes: 38.8%; HIV: 30.2%; Heart disease: 43.8%; Hypertension: 38.6%; Other chronic condition: 35.8%; Sickle cell disease: 42.8%; No known conditions: 28.7%; Unknown (missing response): 30.4%.
  Test item presence against absence among known responses; do not use unknowns
  as the absence group. Check sparse cells and multiple comparisons.
- **Vitals:** The violin and correlation plots nominate heart rate, respiration,
  temperature, and oxygen saturation for comparisons across urgency levels.
  Their Spearman correlations with urgency rank are heart rate
  0.597, respiration 0.392,
  temperature 0.273, and oxygen saturation
  -0.576. Systolic and diastolic correlations are
  0.071 and 0.064; compare
  their plots separately. Missing observations are excluded pairwise, not imputed.
- **Sex and pregnancy:** High/Critical shares by recorded sex are
  Female: 32.3%; Male: 32.3%; pregnancy-response shares are
  Yes: 34.9%; No: 30.2%; Not applicable: 33.6%. Compare composition and sample sizes
  before prioritising demographic hypotheses. Pregnancy response applicability
  differs by recorded sex, so avoid interpreting it as an isolated exposure.
- **Missingness and repeat requests:** The missingness heatmap motivates checking
  whether observation availability differs by urgency. The prior-request count
  distributions are also shown; test any history relationship with patient-level
  dependence accounted for. No plot here establishes that a feature predicts
  unseen patients or that an absent measurement has clinical meaning.

### Data quality surprises and limits

- **Complete grouped-outcome separation:** All 333 requests reporting "Thoughts of self-harm" are High or Critical (192 High; 141 Critical). Audit source label construction and triage-rule provenance before modelling. This is descriptive separation of a grouped outcome, and creates zero cells for later tests.
- Systolic/diastolic missingness patterns are identical:
  **True**. Their missing indicators therefore
  carry duplicate availability information; treat them as a coupled measurement.
- Unknown multi-value responses in this labelled cohort are
  **{'red_flags': 262, 'comorbidities': 293}**, compared with
  **{'red_flags': 267, 'comorbidities': 301}** in all raw intake rows. The denominators
  differ because unlabelled requests are excluded. Unknown remains separate from none.
- Symptom duration units are inferred, not confirmed by metadata. Raw-day skewness
  is 5.099; log-hours skewness is
  -0.229. Minute alignment and broad range support
  the seconds interpretation but cannot prove it.
- Observed ages span 16.03–89.96 years, with
  213 requests under eighteen. This cohort does not represent
  the entire age range of telemedicine users.
- 5,017 labelled requests are beyond the
  first request per patient; patient-level independence would be inappropriate.
  276 unlabelled intake requests are excluded from all
  target comparisons, so label availability could also affect representativeness.
- Preparation found 0 newly
  invalid vital values under its broad technical bounds, and the prepared data has
  0 duplicate request IDs and
  0 negative ages. Passing broad bounds does not confirm
  measurement accuracy; source confirmation of units remains outstanding.

Prioritise the visible severity/pain, red-flag, complaint/onset, duration, age, and
vital relationships for formal testing. Choose tests after checking assumptions,
report effect sizes, handle multiple comparisons and repeated patients, and preserve
patient-grouped time-aware validation for subsequent modelling.
